In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'mimic-cxr-dataset' dataset.
Path to dataset files: /kaggle/input/mimic-cxr-dataset


In [ ]:
import numpy as np
import pandas as pd
data=pd.read_csv("/kaggle/input/mimic-cxr-dataset/mimic_cxr_aug_train.csv")


In [ ]:
data.head()

,Unnamed: 0.1,Unnamed: 0,subject_id,image,view,AP,PA,Lateral,text,text_augment
0,0,0,10000032,['files/p10/p10000032/s50414267/02aa804e-bde0a...,"['PA', 'LATERAL', 'AP']",['files/p10/p10000032/s53911762/68b5c4b1-227d0...,['files/p10/p10000032/s50414267/02aa804e-bde0a...,['files/p10/p10000032/s50414267/174413ec-4ec4c...,"['Findings: There is no focal consolidation, p...","['Findings: There is no focus, pleural effusio..."
1,1,1,10000764,['files/p10/p10000764/s57375967/096052b7-d256d...,"['AP', 'LATERAL']",['files/p10/p10000764/s57375967/096052b7-d256d...,[],['files/p10/p10000764/s57375967/b79e55c3-735ce...,['Findings: PA and lateral views of the chest ...,['Finds: PA and lateral view of the breast int...
2,2,2,10000898,['files/p10/p10000898/s50771383/0c4eb1e1-b8019...,"['LATERAL', 'PA']",[],['files/p10/p10000898/s50771383/2a280266-c8bae...,['files/p10/p10000898/s50771383/0c4eb1e1-b8019...,['Findings: PA and lateral views of the chest ...,['Finds: PA and side view of the chest provide...
3,3,3,10000935,['files/p10/p10000935/s50578979/d0b71acc-b5a62...,"['AP', 'LATERAL', 'LL', 'PA']",['files/p10/p10000935/s50578979/d0b71acc-b5a62...,['files/p10/p10000935/s55697293/c50494f1-90e2b...,['files/p10/p10000935/s51178377/3be619d1-506a6...,['Findings: Lung volumes remain low. There ar...,['Results: Pulmonary volumes remain low. There...
4,4,4,10000980,['files/p10/p10000980/s50985099/6ad03ed1-97ee1...,"['PA', 'LL', 'AP', 'LATERAL']",['files/p10/p10000980/s51967283/943486a3-b3fa9...,['files/p10/p10000980/s50985099/6ad03ed1-97ee1...,['files/p10/p10000980/s54577367/96f9a77c-59b47...,['Findings: Impression: Compared to chest rad...,['Findings: Impression: Compared to the thorac...


In [ ]:
!pip install torch torchvision transformers timm pandas scikit-learn
!pip install nltk rouge-score evaluate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ad71ab70f45106d5c561a34df752480ba1e0514fb828ac2ac487fe17240fdba6
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
#!pip install chexpert-labeler
!pip install radgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.0/588.0 kB 43.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for radgraph: filename=radgraph-0.1.18-py3-none-any.whl size=812635 sha256=6c7cc1d7add4b30c2d6d4e46b4a2afd2e0a76b788d52d130d0ece8eff48a42ee
  Stored in directory: /root/.cache/pip/wheels/fb/3c/fb/214f5d5cdab2a0f9f0904fd81d7fd1134404100b4444554df8
Successfully built radgraph


In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/mimic-cxr-dataset/mimic_cxr_aug_train.csv")
print(df.columns)


Index(['Unnamed: 0.1', 'Unnamed: 0', 'subject_id', 'image', 'view', 'AP', 'PA',
       'Lateral', 'text', 'text_augment'],
      dtype='object')


In [ ]:
import ast
import os

df = data.copy() # Ensure df is populated from the 'data' DataFrame

# Convert the string representation of lists in the 'view' column to actual lists
df['view_list'] = df['view'].apply(ast.literal_eval)

# Filter the DataFrame where 'PA' or 'LATERAL' is in the 'view_list'
df = df[df['view_list'].apply(lambda x: 'PA' in x or 'LATERAL' in x)]

# Drop the temporary 'view_list' column if it exists in the DataFrame
if 'view_list' in df.columns:
    df = df.drop(columns=['view_list'])

# Preprocess 'image' column to create 'image_path'
def get_first_image_full_path(image_list_str, base_path):
    try:
        image_list = ast.literal_eval(image_list_str)
        if image_list:
            return os.path.join(base_path, image_list[0])
    except (ValueError, SyntaxError):
        pass # Handle cases where string is not a valid list representation
    return None

df['image_path'] = df['image'].apply(lambda x: get_first_image_full_path(x, path))

# Drop rows where image_path could not be determined (e.g., empty lists or invalid strings)
df = df.dropna(subset=['image_path'])

# Add an extra filter to ensure the file actually exists on disk
df = df[df['image_path'].apply(os.path.exists)]

In [ ]:
def split_sections(txt):
    f = ""
    i = ""
    if "FINDINGS:" in txt:
        f = txt.split("FINDINGS:")[1].split("IMPRESSION:")[0]
    if "IMPRESSION:" in txt:
        i = txt.split("IMPRESSION:")[1]
    return f, i


In [ ]:
from torchvision import transforms

img_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class CXRDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tk = tokenizer

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        img = img_tf(img)

        text = row["text"]

        tokens = self.tk(
            text,
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        return img, tokens.input_ids.squeeze()

    def __len__(self):
        return len(self.df)


In [ ]:
from transformers import ViTModel

vit = ViTModel.from_pretrained("google/vit-base-patch16-224")


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5 = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
import torch
def pro_fa_features(vit_out):
    cls_token = vit_out.last_hidden_state[:,0]      # organ level
    patches = vit_out.last_hidden_state[:,1:]       # region level
    region_feat = patches.mean(dim=1)

    return torch.cat([cls_token, region_feat], dim=1)


In [ ]:
import torch.nn as nn

class MIX_MLP(nn.Module):
    def __init__(self, dim, n_labels):
        super().__init__()
        self.pathA = nn.Sequential(
            nn.Linear(dim, dim//2),
            nn.ReLU(),
            nn.Linear(dim//2, n_labels)
        )
        self.pathB = nn.Sequential(
            nn.Linear(dim, dim*2),
            nn.ReLU(),
            nn.Linear(dim*2, n_labels)
        )

    def forward(self,x):
        return self.pathA(x) + self.pathB(x)


In [ ]:
attn = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

def rcta(image_feat, text_feat, label_feat):

    c1,_ = attn(image_feat, text_feat, text_feat)
    c2,_ = attn(c1, label_feat, label_feat)
    verified,_ = attn(c2, image_feat, image_feat)

    return verified


In [ ]:
class ReportModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = vit
        self.mix = MIX_MLP(1536, 14)
        self.t5 = t5
        self.proj = nn.Linear(1536, 512)

    def forward(self, images, input_ids):

        v = self.vit(images)
        feat = pro_fa_features(v)

        labels = self.mix(feat)

        enc = self.proj(feat).unsqueeze(1)

        out = self.t5(
            encoder_outputs=(enc,),
            labels=input_ids
        )

        return out.loss, labels


In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn

# Check if the DataFrame is empty before proceeding
if len(df) == 0:
    print("Warning: The DataFrame 'df' is empty after preprocessing. Cannot create DataLoader and proceed with training.")
    # The training loop will be skipped if df is empty.
else:
    # Instantiate the dataset and data loader
    dataset = CXRDataset(df, tokenizer) # Assuming 'df' and 'tokenizer' are defined from previous cells
    loader = DataLoader(dataset, batch_size=4, shuffle=True) # Define your batch size as needed

    # Instantiate the model
    model = ReportModel() # Assuming ReportModel, vit, t5, MIX_MLP are defined

    # Define loss function
    bce = nn.BCEWithLogitsLoss() # Binary Cross-Entropy Loss for multi-label classification

    # Define optimizer
    opt = optim.Adam(model.parameters(), lr=1e-4) # Adam optimizer with a learning rate

    # Placeholder for ground truth labels. This should come from your dataset.
    # For now, let's create a dummy tensor for demonstration, assuming 14 labels as per MIX_MLP
    gt_labels = torch.randint(0, 2, (4, 14)).float() # Example: batch_size x num_labels

    # Training loop
    for imgs, ids in loader:

        # Move data to the appropriate device (e.g., GPU if available)
        # imgs = imgs.to(device)
        # ids = ids.to(device)
        # gt_labels = gt_labels.to(device)

        opt.zero_grad()

        loss_text, label_logits = model(imgs, ids)

        loss_cls = bce(label_logits, gt_labels) # Use the correct ground truth labels from your dataset

        loss = loss_text + 0.5 * loss_cls

        loss.backward()
        opt.step()

    print("Training loop executed without NameError (further setup for data and device might be needed).")

In [ ]:
import torch
from PIL import Image
from transformers.modeling_outputs import BaseModelOutput # Import BaseModelOutput

# To generate a report, you first need to pass an image through the model's encoder
# (ViT and projection layers) to obtain the 'enc' variable.

# Check if the DataFrame 'df' is empty. If so, we cannot process real images.
if len(df) == 0:
    print("Warning: The DataFrame 'df' is empty. Using a dummy image input for demonstration.")
    dummy_image_input = torch.randn(1, 3, 224, 224) # Batch size 1, 3 channels, 224x224
else:
    # In a real scenario, you would load and preprocess an actual image from df:
    # For this example, let's take the first image if df is not empty.
    print("Processing the first image from the DataFrame for generation.")
    try:
        img_path = df.iloc[0]['image_path']
        single_image = img_tf(Image.open(img_path).convert("RGB")).unsqueeze(0)
        dummy_image_input = single_image
    except Exception as e:
        print(f"Error loading actual image: {e}. Falling back to dummy input.")
        dummy_image_input = torch.randn(1, 3, 224, 224)

# Pass the image through the ViT encoder
v_output = model.vit(dummy_image_input)

# Process ViT output to get features
feat = pro_fa_features(v_output)

# Project features to get 'enc' for T5 encoder_outputs
enc = model.proj(feat).unsqueeze(1)

# Wrap 'enc' in a BaseModelOutput object to mimic the encoder output structure expected by generate
dummy_encoder_outputs = BaseModelOutput(last_hidden_state=enc)

gen = model.t5.generate(
    encoder_outputs=dummy_encoder_outputs, # Pass the wrapped encoder outputs
    max_length=200
)

report = tokenizer.decode(gen[0], skip_special_tokens=True)

print("Generated Report:", report)

Generated Report: griffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgriffgr

In [ ]:
import pandas as pd
# df = pd.read_csv("/kaggle/input/mimic-cxr-dataset/mimic_cxr_aug_train.csv") # Commented out to prevent overwriting processed df
print(df.columns)
print(df.head(2))

Index(['Unnamed: 0.1', 'Unnamed: 0', 'subject_id', 'image', 'view', 'AP', 'PA',
       'Lateral', 'text', 'text_augment'],
      dtype='object')
   Unnamed: 0.1  Unnamed: 0  subject_id  \
0             0           0    10000032   
1             1           1    10000764   

                                               image                     view  \
0  ['files/p10/p10000032/s50414267/02aa804e-bde0a...  ['PA', 'LATERAL', 'AP']   
1  ['files/p10/p10000764/s57375967/096052b7-d256d...        ['AP', 'LATERAL']   

                                                  AP  \
0  ['files/p10/p10000032/s53911762/68b5c4b1-227d0...   
1  ['files/p10/p10000764/s57375967/096052b7-d256d...   

                                                  PA  \
0  ['files/p10/p10000032/s50414267/02aa804e-bde0a...   
1                                                 []   

                                             Lateral  \
0  ['files/p10/p10000032/s50414267/174413ec-4ec4c...   
1  ['files/p10/p10000764/s57375

In [ ]:
from PIL import Image

for p in df["image_path"].head(3):
    img = Image.open(p)
    print(img.size)


KeyError: 'image_path'